# R3. Budget cases (30 min, teams of 2 to 3)
Each team draws a **case card**. You must **predict the cost first**, then run a sample, then compare. The size of your estimation error is part of the score.

**Steps:** 1) read the card, 2) estimate tokens and cost for the full job, 3) run a sample, 4) pull levers to fit the budget, 5) log your best run with your prediction in the note.

**How the arena works**
- Your prompt can use placeholders: `{text}` (the item), `{labels}`, `{codebook}` (the official definitions), `{schema}` (extraction).
- The harness adds one output rule so everyone is parsed the same way. You do **not** need to ask for a specific format.
- Every run costs real money from your allowance and appears on the board. **Write your hypothesis in `note=` before you run.**
- The hidden test set stays with the instructor. Scores you see here are on the *dev* set.

In [ ]:
# ===== Run this cell first =====
import os, sys, hashlib, json
KIT_URL = "https://github.com/atsizelti/prompt_engineering.git"   # public repo with the kit
KIT_DIR = "arena_kit/prompt_arena_kit"                            # folder inside the clone
try:
    import arena
except ImportError:
    if os.path.exists("arena_kit"):
        os.system("git -C arena_kit pull -q")            # refresh a copy cloned earlier in this session
    else:
        os.system(f"git clone -q {KIT_URL} arena_kit")
    sys.path.insert(0, KIT_DIR)
    import arena

NICK      = "your-nickname"      # pick a nickname (no real name needed)
PROVIDER  = "openai"             # "openai", or "mock" (practice with no key)
ENDPOINT  = "https://script.google.com/macros/s/AKfycbxdVYynFuPHaHg73kTFzL5R0TYXrPA8Tr0adMEsb0LKUS5vpDgysktWLPTTcLWrS7Jz/exec"     # the class board (already filled in: you do not need to change it)
TOKEN     = "f-BEqiAfHhAc2O-b"
ALLOWANCE = 0.06                # dollars you may spend (the whole class shares one small budget; re-running this cell does not reset it)

if os.environ.get("ARENA_MOCK"):
    PROVIDER = "mock"
if os.environ.get("ARENA_NO_BOARD"):                  # used by the instructor's offline tests
    ENDPOINT = TOKEN = ""
KEY = None
if PROVIDER != "mock":
    try:
        from google.colab import userdata
        KEY = userdata.get("OPENAI_API_KEY")
    except Exception:
        KEY = os.environ.get("OPENAI_API_KEY")
try:                                                  # the board address and password, set once as Colab secrets
    from google.colab import userdata
    ENDPOINT = ENDPOINT or userdata.get("BOARD_ENDPOINT")
    TOKEN = TOKEN or userdata.get("BOARD_TOKEN")
except Exception:
    ENDPOINT = ENDPOINT or os.environ.get("BOARD_ENDPOINT", "")
    TOKEN = TOKEN or os.environ.get("BOARD_TOKEN", "")
ENDPOINT, TOKEN = "".join(ENDPOINT.split()), "".join(TOKEN.split())   # remove hidden spaces and line breaks from pasting
arena.setup(NICK, provider=PROVIDER, api_key=KEY, endpoint=ENDPOINT or None,
            token=TOKEN or None, allowance_usd=ALLOWANCE)
arena.set_round("R3")

In [ ]:
CASE = "municipal"      # "municipal", "codebook", "cascade", "cold_start", "audit"   (your team draws one)
CASES = {
 "municipal":  dict(task="B1", n_items=30000, budget=3.0,  goal="Maximise macro-F1 for 30,000 Turkish citizen complaints within $3."),
 "codebook":   dict(task="B2", n_items=500,   budget=40.0, goal="Reach kappa >= 0.7 on 500 parliamentary speeches (12-category codebook) within $40."),
 "cascade":    dict(task="B4", n_items=20000, budget=15.0, goal="Best F1 per dollar for 20,000 survey answers: a cheap model first, a strong model only when unsure."),
 "cold_start": dict(task="B3", n_items=2000,  budget=10.0, goal="No examples. Write the best prompt from the codebook alone. ONE scored dev run."),
 "audit":      dict(task="F",  n_items=0,     budget=10.0, goal="Measure whether labels flip when only the defendant's name changes. Report the flip rate."),
}
C = CASES[CASE]
print(f"CASE: {CASE}\n{C['goal']}\nBudget ${C['budget']}  |  items {C['n_items']}")
if C["n_items"]:
    print(f"=> maximum affordable cost per item: ${C['budget']/C['n_items']:.5f}  (per 1,000 items: ${C['budget']/C['n_items']*1000:.2f})")
arena.peek(C["task"], k=3)

### Step 2. Estimate BEFORE you run
Guess tokens per item (input and output) and project the bill. Write your numbers down.

In [ ]:
TOKENS_IN_GUESS, TOKENS_OUT_GUESS = 350, 20      # <- your guess per item. Long codebook? Add it. Reasoning text? Raise the output guess.
est = arena.estimate_cost(TOKENS_IN_GUESS, TOKENS_OUT_GUESS, C["n_items"] or 1000)
print(f"Projected for {est['n_items']} items on {est['model']}: ${est['projected_usd']:.2f}  (${est['per_1k_items']:.2f} per 1,000)")

### Step 3. Run a sample and compare

In [ ]:
PROMPT = """Classify the text.
Options: {labels}

Codebook:
{codebook}

Text: {text}"""
PRED_PER_1K = est["per_1k_items"]
res = arena.evaluate(PROMPT, task=C["task"], n=24, condition=CASE,
                     note=f"case={CASE}; predicted_cost_per_1k={PRED_PER_1K:.3f}; H: baseline")
res.summary()
full = res.metrics["cost_per_1k"] * (C["n_items"] or 1000) / 1000
print(f"\nActual projected cost for the full job: ${full:.2f}  vs budget ${C['budget']}")
print(f"Estimation error: {abs(res.metrics['cost_per_1k']-PRED_PER_1K)/max(PRED_PER_1K,1e-9):.0%}")
print("WITHIN BUDGET" if full <= C["budget"] else "OVER BUDGET: pull a lever")

### Step 3b. The model ladder
The **same prompt** on three models. Compare quality, cost, and *how* they fail. **Watch the cost column:** the model with the lowest *price per token* is not necessarily the cheapest *per item*. Some models spend hidden reasoning tokens. (The strongest model is not allowed in class; the instructor may demo it.)

In [ ]:
for mdl in ["gpt-5-nano", "gpt-4o-mini", "gpt-5.4-mini"]:
    r = arena.evaluate(PROMPT, task=C["task"], n=24, model=mdl, condition=CASE, note=f"case={CASE}; model ladder: {mdl}")
    print(f"{mdl:<14} F1 {r.metrics['macro_f1']:.2f}   cost per 1,000 items ${r.metrics['cost_per_1k']:.2f}\\n")

### Step 4. Levers to try (one at a time, with a hypothesis)
| Lever | Try |
|---|---|
| **Model tier** | `model="..."` (see prices.json: small, mid, large) |
| **Prompt length** | Drop few-shots; include only the codebook lines that matter |
| **Prompt order for caching** | Put stable instructions and examples **first**, the changing item **last** (already how `{text}` works) |
| **Batching** | Several items per call: lower per-item overhead, but watch accuracy |
| **Output length** | Ask for the label only, no reasoning |
| **Cascade** | Cheap model first, strong model for low-confidence items (cell below) |
| **Context rot** (codebook case) | Compare the full 12-category codebook against only the relevant sections |

In [ ]:
# Example lever: a cheaper, shorter prompt. Change one thing and write the hypothesis.
PROMPT_SHORT = "Label the text with one of: {labels}\n\nText: {text}"
res_b = arena.evaluate(PROMPT_SHORT, task=C["task"], n=24, condition=CASE, note=f"case={CASE}; H: dropping the codebook saves cost with small F1 loss")
res_b.summary()

In [ ]:
# CASCADE case: the cheap model labels everything; the strong model sees only low-confidence items.
CHEAP, STRONG = "gpt-4o-mini", "gpt-5.4-mini"      # allowed: gpt-5-nano, gpt-4o-mini, gpt-5.4-nano, gpt-5.4-mini
for th in (50, 70, 90):
    r = arena.cascade(PROMPT, task=C["task"], cheap=CHEAP, strong=STRONG, threshold=th, n=24, note=f"case={CASE}; threshold={th}")
    print(f"threshold {th}: F1 {r.metrics['macro_f1']:.3f}  cost/1k ${r.metrics['cost_per_1k']:.2f}\n")

In [ ]:
# AUDIT case: do labels flip when only the name changes?
r = arena.evaluate("Rate the severity of the offence from the facts only.\nOptions: {labels}\n\n{codebook}\n\nText: {text}",
                   task="F", condition="audit", note="case=audit; H: severity depends only on facts")
r.summary()
print(f"Flip rate across matched pairs: {r.metrics.get('flip_rate', float('nan')):.0%}. Correct is 0%.")

### Team reflection
Which lever bought the most F1 per dollar? What would you change if the budget were cut in half?